# CS6493 Tutorial 11: LangChain


- Introduction to LangChain

- Create a ChatBot with LangChain + Ollama

- Create a Retrieval Augmented ChatBot with LangChain + Ollama

- Practice: Conversation Retrieval Chain


## Introduction to LangChain

LangChain is a framework for developing applications powered by language models. It enables applications that:

- Are **context-aware**: connect a language model to sources of context (prompt instructions, few shot examples, content to ground its response in, etc.)

- **Reason**: rely on a language model to reason (about how to answer based on provided context, what actions to take, etc.)

This framework consists of several parts.

- **LangChain Libraries:** The Python and JavaScript libraries. Contains interfaces and integrations for a myriad of components, a basic run time for combining these components into chains and agents, and off-the-shelf implementations of chains and agents.
- **LangChain Templates:** A collection of easily deployable reference architectures for a wide variety of tasks.
- **LangServe:** A library for deploying LangChain chains as a REST API.
- **LangSmith:** A developer platform that lets you debug, test, evaluate, and monitor chains built on any LLM framework and seamlessly integrates with LangChain.

Together, these products simplify the entire application lifecycle:

- **Develop:** 88 Write your applications in LangChain/LangChain.js. Hit the ground running using Templates for reference.
- **Productionize:** Use LangSmith to inspect, test and monitor your chains, so that you can constantly improve and deploy with confidence.
- **Deploy:** Turn any chain into an API with LangServe.

LangChain Libraries

The main value props of the LangChain packages are:

- **Components:** composable tools and integrations for working with language models. Components are modular and easy-to-use, whether you are using the rest of the LangChain framework or not
- **Off-the-shelf chains:** built-in assemblages of components for accomplishing higher-level tasks

Off-the-shelf chains make it easy to get started. Components make it easy to customize existing chains and build new ones.

The LangChain libraries themselves are made up of several different packages.

- **langchain-core:** Base abstractions and LangChain Expression Language.
- **langchain-community:** Third party integrations.
- **langchain:** Chains, agents, and retrieval strategies that make up an application's cognitive architecture.

## Create a ChatBot with LangChain + Ollama

LangChain enables building application that connect external sources of data and computation to LLMs. In this quickstart, we will walk through a few different ways of doing that. We will start with a simple LLM chain, which just relies on information in the prompt template to respond. Next, we will build a retrieval chain, which fetches data from a separate database and passes that into the prompt template. We will then add in chat history, to create a conversation retrieval chain. This allows you to interact in a chat manner with this LLM, so it remembers previous questions.

Install

In [1]:
!pip install langchain==0.3.21

In [2]:
!pip install langchain-community==0.3.20

We can use models available via API, like OpenAI, and local open source models, using integrations like Ollama. Here we choose to use Ollama.

First, let's install Ollama and start Ollama service

In [3]:
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
!setsid nohup ollama serve > run_serve.txt 2>&1 & # we must use this command to make sure the ollama serve is running

In [5]:
# check models we can use
!ollama list

NAME        ID              SIZE      MODIFIED      
gemma:2b    b50d6c999e59    1.7 GB    5 minutes ago    


We should firstly choose a model as the backbone model for our application. To reduce the cost of computation resources, here we just choose gemma-2b model as our backbone model.

In [6]:
!setsid ollama run gemma:2b > run_gemma_2b.txt 2>&1 &

In [7]:
from langchain_community.llms import Ollama
llm = Ollama(model="gemma:2b")

/tmp/ipykernel_5047/3720262705.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gemma:2b")


Once you've installed and initialized the LLM of your choice, we can try using it! Let's ask it what LangSmith is - this is something that wasn't present in the training data so it shouldn't have a very good response.

In [8]:
llm.invoke("how can langsmith help with testing?")

'**Langsmith** can be a valuable tool for testing, particularly for the following reasons:\n\n**1. Code Coverage:**\n* Langsmith provides comprehensive coverage analysis, highlighting areas where your code is tested.\n* This helps identify gaps in your testing efforts and ensures that important functionalities are thoroughly tested.\n\n**2. Test Case Generation:**\n* Langsmith can generate test cases automatically based on your code structure and API specifications.\n* This eliminates the need for manual test case creation and reduces the effort required for testing.\n\n**3. Defect Prioritization:**\n* By analyzing the generated test cases, you can prioritize defects based on their severity and likelihood.\n* This allows you to focus your testing efforts on areas that matter most.\n\n**4. Code Quality Checks:**\n* Langsmith can identify and report potential code quality issues, such as unused variables, improper variable names, and logical errors.\n* This helps improve the quality of y

We can also guide its response with a prompt template. Prompt templates convert raw user input to better input to the LLM.

In [9]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are world class technical documentation writer."),
    ("user", "{input}")
])

We can now combine these into a simple LLM chain:

In [10]:
chain = prompt | llm

We can now invoke it and ask the same question. It still won't know the answer, but it should respond in a more proper tone for a technical writer!

In [11]:
chain.invoke({"input": "how can langsmith help with testing?"})

'**Langsmith can help with testing by:**\n\n**1. Identifying Defects:**\n* Analyzing and interpreting technical documents to identify areas for testing.\n* Detecting inconsistencies, gaps, and missing information.\n* Prioritizing defects based on severity and impact.\n\n**2. Generating Test Cases:**\n* Creating comprehensive test cases that cover different functionalities and scenarios.\n* Defining test steps, expected results, and pass/fail criteria.\n* Generating test cases from user stories and requirements.\n\n**3. Automating Testing:**\n* Creating automated tests using tools like Selenium and JUnit.\n* Reducing manual effort and increasing efficiency.\n* Identifying areas for automation and creating test scripts.\n\n**4. Analyzing Test Results:**\n* Collecting and interpreting test data.\n* Identifying defects, passing tests, and troubleshooting issues.\n* Reporting on test results and providing insights to stakeholders.\n\n**5. Collaboration and Communication:**\n* Facilitating c

The output of a ChatModel (and therefore, of this chain) is a message. However, it's often much more convenient to work with strings. Let's add a simple output parser to convert the chat message to a string.

In [12]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

We can now add this to the previous chain:

In [13]:
chain = prompt | llm | output_parser

We can now invoke it and ask the same question. The answer will now be a string (rather than a ChatMessage).

In [14]:
chain.invoke({"input": "how can langsmith help with testing?"})

"**Langsmith can help with testing in several ways:**\n\n**1. Identifying and prioritizing testing tasks:**\n\n* Langsmith's powerful natural language processing (NLP) capabilities can analyze vast amounts of technical documentation and identify key topics, sections, and defects.\n* This allows you to prioritize testing efforts and focus on areas that are most likely to impact the final product.\n\n**2. Automating testing tasks:**\n\n* Langsmith can automate repetitive tasks such as data entry, report generation, and test case creation.\n* This frees up your time to focus on more complex and strategic testing activities.\n\n**3. Generating test cases:**\n\n* Langsmith can generate comprehensive test cases based on the documentation you provide.\n* These tests can be automatically run, ensuring that you are testing the most important features and edge cases.\n\n**4. Identifying potential defects:**\n\n* By analyzing the documentation and identifying patterns in the language used, Langsm

## Retrieval Chain

To properly answer the original question ("how can langsmith help with testing?"), we need to provide additional context to the LLM. We can do this via retrieval. Retrieval is useful when you have too much data to pass to the LLM directly. You can then use a retriever to fetch only the most relevant pieces and pass those in.

In this process, we will look up relevant documents from a Retriever and then pass them into the prompt. A Retriever can be backed by anything - a SQL table, the internet, etc - but in this instance we will populate a vector store and use that as a retriever. For more information on vectorstores, see this documentation.

First, we need to load the data that we want to index. To do this, we will use the WebBaseLoader. This requires installing BeautifulSoup:

In [15]:
!pip install beautifulsoup4==4.13.3

After that, we can import and use WebBaseLoader.

In [16]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.smith.langchain.com/user_guide")

docs = loader.load()

Next, we need to index it into a vectorstore. This requires a few components, namely an embedding model and a vectorstore.

For embedding models, we once again provide examples for accessing via API or by running local models.

In [17]:
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="gemma:2b")

/tmp/ipykernel_5047/2283036239.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="gemma:2b")


Now, we can use this embedding model to ingest documents into a vectorstore. We will use a simple local vectorstore, FAISS, for simplicity's sake.

First we need to install the required packages for that:

In [18]:
!pip install faiss-cpu==1.10.0

Then we can build our index:

In [19]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(docs)
vector = FAISS.from_documents(documents, embeddings)

Now that we have this data indexed in a vectorstore, we will create a retrieval chain. This chain will take an incoming question, look up relevant documents, then pass those documents along with the original question into an LLM and ask it to answer the original question.

First, let's set up the chain that takes a question and the retrieved documents and generates an answer.

In [20]:
from langchain.chains.combine_documents import create_stuff_documents_chain

prompt = ChatPromptTemplate.from_template("""Answer the following question based only on the provided context:

<context>
{context}
</context>

Question: {input}""")

document_chain = create_stuff_documents_chain(llm, prompt)

If we wanted to, we could run this ourselves by passing in documents directly:

In [21]:
from langchain_core.documents import Document

document_chain.invoke({
    "input": "how can langsmith help with testing?",
    "context": [Document(page_content="langsmith can let you visualize test results")]
})

'The context does not provide any information about how langsmith can help with testing, so I cannot answer this question from the provided context.'

However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [22]:
from langchain.chains import create_retrieval_chain

retriever = vector.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)

We can now invoke this chain. This returns a dictionary - the response from the LLM is in the answer key

In [23]:
response = retrieval_chain.invoke({"input": "how can langsmith help with testing?"})
print(response["answer"])

The context does not provide information about how Langsmith can help with testing, so I cannot answer this question from the provided context.


# Practice: Conversation Retrieval Chain

The chain we've created so far can only answer single questions. One of the main types of LLM applications that people are building are chat bots. So how do we turn this chain into one that can answer follow up questions?

We can still use the create_retrieval_chain function, but we need to change two things:

-The retrieval method should now not just work on the most recent input, but rather should take the whole history into account.

-The final LLM chain should likewise take the whole history into account

Updating Retrieval

In order to update retrieval, we will create a new chain. This chain will take in the most recent input (input) and the conversation history (chat_history) and use an LLM to generate a search query.

In [24]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

# First we need a prompt that we can pass into an LLM to generate this search query

prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    ("user", "Given the above conversation, generate a search query to look up to get information relevant to the conversation")
])
retriever_chain = create_history_aware_retriever(llm, retriever, prompt)

We can test this out by passing in an instance where the user asks a follow-up question.

In [25]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [HumanMessage(content="Can LangSmith help test my LLM applications?"), AIMessage(content="Yes!")]
retriever_chain.invoke({
    "chat_history": chat_history,
    "input": "Tell me how"
})

[Document(id='8f4b13fe-7714-4ebd-b553-e95bfcf29181', metadata={'source': 'https://docs.smith.langchain.com/user_guide', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='To see how to create a service key or Personal Access Token, see the setup guide\n\u200bOrganization roles\nOrganization roles are distinct from the Enterprise feature workspace RBAC and are used in the context of multiple workspaces. Your organization role determines your workspace membership characteristics and your organization-level permissions.\nThe organization role selected also impacts workspace membership as described here:\n\nOrganization Admin grants full access to manage all organization configuration, users, billing, and workspaces.\n\nAn Organization Admin has Admin access to all workspaces in an organization.\n\n\nOrganization User may read organization information but cannot execute any write actions at the organization level. An Organization User may create Personal Access Token

You should see that this returns documents about testing in LangSmith. This is because the LLM generated a new query, combining the chat history with the follow-up question.

Now that we have this new retriever, we can create a new chain to continue the conversation with these retrieved documents in mind.

In [26]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's questions based on the below context:\n\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
])
document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriever_chain, document_chain)

We can now test this out end-to-end:

In [27]:
chat_history = [HumanMessage(content="Can LangSmith help test my LLM applications?"), AIMessage(content="Yes!")]
retrieval_chain.invoke({
    "chat_history": chat_history,
    "input": "Tell me how"
})

{'chat_history': [HumanMessage(content='Can LangSmith help test my LLM applications?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Yes!', additional_kwargs={}, response_metadata={})],
 'input': 'Tell me how',
 'context': [Document(id='e0cc1bad-3bfa-48f0-a58a-b53f0a961955', metadata={'source': 'https://docs.smith.langchain.com/user_guide', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='If you have questions or concerns about our pricing model, please feel free to contact support via support.langchain.com and let us know your thoughts!\nHow does data retention affect downstream features?\n\nAnnotation Queues, Run Rules, and Feedback: Traces that use these features will be auto-upgraded.\nMonitoring: The monitoring tab will continue to work even after a base tier trace’s data retention period ends. It is powered by trace metadata that exists for >30 days, meaning that your monitoring graphs will continue to stay accurate even on base tier t